# 01 · Explore the OpenAlex ALAN papers

Matches the title and abstract of each paper against the word lists in `vocab/terms.py`, flags papers that are off-topic for the biodiversity focus (human health), and saves a table of the results.

**Reads:** `data/raw/openalex_alan.csv` (made by `src/fetch_openalex.py`), `kg/seed_statements.csv`, `kg/items.csv`
**Writes:** `data/output/table_abstracts.csv` (git-ignored)

**After editing `vocab/terms.py`:** click Restart, then Run All. Python keeps the old word lists in memory otherwise.

## Progress log

| Date | Change | Papers with no match | In-scope papers with no match |
|---|---|---|---|
| 17 Nov | Round 1 baseline | 949 of 2,227 | not measured yet |
| 17 Nov | + zebrafish, elasmobranch, shark | 938 | not measured yet |
| 17 Nov | + seabird, petrel, Procellariiformes | 937 | 452 of 1,576 (simple flag) |
| 18 Nov | + Drosophila, deer, mouse, mice, rat, lizard, anole, frog, amphipod, chipmunk, epiphyte, wildlife | 804 | 385 of 1,666 (simple flag) |

3-way scope after the 18 Nov terms: in scope 1,666 · off-topic 451 · unsure 110.

Add a row here after each round of word-list changes.

## 1. Setup: load the papers and define the matcher

In [1]:
import re
import sys
import pandas as pd

sys.path.append("..")
from vocab import terms

df = pd.read_csv("../data/raw/openalex_alan.csv")
df["text"] = df["title"].fillna("") + ". " + df["abstract"].fillna("")

`make_pattern` builds a whole-word regex for each term. Abbreviations (SQM, PAR, lx) must match exactly. Other terms ignore case and also match a simple plural ("moth" matches "moths").

A term has to start at a word boundary, so "bird" does not match inside "seabird" and "fish" does not match inside "zebrafish": list compounds as their own terms. Irregular plurals need their own entry too ("mouse" and "mice").

In [2]:
def make_pattern(term):
    # Abbreviations like SQM, PAR, W must match exactly (capital letters).
    case_sensitive = term.isupper() or term == "lx"
    if case_sensitive:
        return re.compile(r"(?<!\w)" + re.escape(term) + r"(?!\w)")
    # Other terms ignore case and also match a plural ("moth" -> "moths").
    return re.compile(r"(?<!\w)" + re.escape(term) + r"(?:e?s)?(?!\w)", re.IGNORECASE)


def find_terms(text, term_list):
    found = [term for term in term_list if make_pattern(term).search(text)]
    return "; ".join(found)

## 2. Round 1 matching

One new column per category holds the terms found in each paper's title and abstract. `no_match` is True when a paper matches nothing in any of the five categories.

In [3]:
categories = {
    "methods":     terms.METHODS_INSTRUMENTS,
    "quantities":  terms.QUANTITIES,
    "units":       terms.UNITS,
    "disciplines": terms.DISCIPLINES,
    "species":     terms.TAXA,
}

# One new column per category: the terms found in each paper
for column, term_list in categories.items():
    df[column] = df["text"].apply(lambda t: find_terms(t, term_list))

# True for papers where nothing matched in any of the five categories
no_match = (df[list(categories)] == "").all(axis=1)
print(no_match.sum(), "of", len(df), "papers have no match")

637 of 2227 papers have no match


## 3. Scope: is the paper about biodiversity?

The project follows the Hölker et al. 2021 "11 pressing research questions" paper (biodiversity), so human-health papers are off-topic, unless they are about animals.

**Simple flag (first cell below):** human-health word AND no species match. A flagged paper can still match another category (unit, quantity), so flagged and unmatched overlap.

**3-way `scope` (second cell, refines the simple flag):**
- `off-topic`: flagged, with 2 or more health terms or one strong term (cancer, patient, diabetes, ...)
- `unsure`: flagged on a single weak term (sleep, participant, clinical, ...)
- `in scope`: everything else

In [4]:
df["human_health"] = df["text"].apply(lambda t: find_terms(t, terms.HUMAN_HEALTH))

off_topic = (df["human_health"] != "") & (df["species"] == "")
in_scope = ~off_topic

print(off_topic.sum(), "papers flagged off-topic (human health, no species)")
print((no_match & in_scope).sum(), "of", in_scope.sum(), "in-scope papers have no match")

595 papers flagged off-topic (human health, no species)
274 of 1632 in-scope papers have no match


In [5]:
strong = ["cancer", "cardiovascular", "obesity", "diabetes", "patient",
          "COVID", "depression", "mental health", "insomnia", "epidemiology"]

n_terms = df["human_health"].apply(lambda s: len(s.split("; ")) if s else 0)
strong_hits = df["human_health"].apply(
    lambda s: sum(t in strong for t in s.split("; ")))

flagged = (df["human_health"] != "") & (df["species"] == "")

df["scope"] = "in scope"
df.loc[flagged & ((n_terms >= 2) | (strong_hits >= 1)), "scope"] = "off-topic"
df.loc[flagged & (n_terms == 1) & (strong_hits == 0), "scope"] = "unsure"

df["scope"].value_counts()

scope
in scope     1632
off-topic     482
unsure        113
Name: count, dtype: int64

## 4. Summary numbers

In scope = all papers minus flagged. The flagged group and the no-match group overlap, so the table splits each of them.

In [6]:
n = len(df)
matched = ~no_match

summary = pd.DataFrame({
    "group": [
        "All papers",
        "Match at least one category",
        "Match nothing",
        "Flagged off-topic (human health, no species)",
        "   flagged AND match nothing",
        "   flagged BUT match something else",
        "In scope (not flagged)",
        "   in scope AND match nothing",
        "   in scope AND match something",
    ],
    "papers": [
        n,
        matched.sum(),
        no_match.sum(),
        off_topic.sum(),
        (off_topic & no_match).sum(),
        (off_topic & matched).sum(),
        in_scope.sum(),
        (in_scope & no_match).sum(),
        (in_scope & matched).sum(),
    ],
})
summary["% of all"] = (summary["papers"] / n * 100).round(1)
summary

,group,papers,% of all
0,All papers,2227,100.0
1,Match at least one category,1590,71.4
2,Match nothing,637,28.6
3,"Flagged off-topic (human health, no species)",595,26.7
4,flagged AND match nothing,363,16.3
5,flagged BUT match something else,232,10.4
6,In scope (not flagged),1632,73.3
7,in scope AND match nothing,274,12.3
8,in scope AND match something,1358,61.0


## 5. Checking the flag

Reading samples to see whether the flag removes papers it should not. Look for animals, ecology and light-measurement papers.

In [7]:
df[off_topic][["title", "human_health"]].sample(15, random_state=2)

,title,human_health
1806,Oxidative Stress and Semen Quality Among Night...,patient; participant; clinical; shift work; ox...
2204,Preventive strategies in neuroimmunology,patient; obesity; sleep
1025,The Relationship Between Breast Cancer Risk an...,cancer; sleep; shift work; children
969,Gut jet lag: how circadian rhythm disruption u...,clinical
1658,The impact of pollution on mental health: A na...,anxiety; mental health; cognitive; neurodegene...
963,Residential-area nighttime light exposure and ...,participant
314,Digital Endocrine Disruptors: a Concept Note o...,endocrine disruptor
2005,Blue light as a performance enhancer: Impacts ...,participant; sleep; retinal; cognitive
1963,"Myopia development: multifactorial interplay, ...",clinical; retinal
1226,Time-restricted eating as a potential strategy...,clinical; glucose; cognitive


In [8]:
n_terms = df["human_health"].apply(lambda s: len(s.split("; ")) if s else 0)
one_term = off_topic & (n_terms == 1)

print(one_term.sum(), "of", off_topic.sum(), "flagged on a single term")
df[one_term]["human_health"].value_counts()

151 of 595 flagged on a single term


human_health
participant            26
sleep                  24
oxidative stress       14
cognitive              12
cancer                 11
cardiovascular         10
clinical                6
cohort                  5
glucose                 3
wearable                3
anxiety                 3
women                   3
retinal                 3
patient                 3
obesity                 3
diabetes                2
COVID                   2
mental health           2
children                2
epidemiology            2
childhood               2
depression              2
adolescent              2
neonatal                1
shift work              1
insomnia                1
ergonomic               1
endocrine disruptor     1
neurodegenerative       1
Name: count, dtype: int64

In [9]:
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 200)

unsure = df[df["scope"] == "unsure"][["title", "human_health"]]
unsure.sort_values("human_health")

,title,human_health
1155,"The independent and joint associations of community space environmental satisfaction, and parent...",adolescent
270,Long-term outdoor artificial light at night and psychological symptoms among Chinese adolescents...,adolescent
1537,Pedestrians’ alertness and perceived environmental safety under non-uniform urban lighting,anxiety
549,Ligustilide alleviates the mitophagy and anxiety-like behavior produced by nighttime lighting,anxiety
90,Association between artificial light at night and the incidence of anxiety: a global ecological ...,anxiety
1852,Appropriate Lifelong Circadian Rhythms Are Established During Infancy: A Narrative Review,childhood
464,Reclaiming the Right to Starlight: A Human Rights Perspective on Light Pollution and Urban Crimi...,childhood
168,Reducing artificial light at night exposure in the urban environment could decrease light pollut...,children
1070,Measuring the visual environment of children and young people at risk of myopia: a scoping review,children
969,Gut jet lag: how circadian rhythm disruption undermines the Chrono-Microbiota-Motility axis and ...,clinical


**Findings from reading the unsure list (18 Nov, when it had 151 papers):**
- About 23 wildlife and ecology papers were hidden only because their animal was missing from `TAXA` (fixed by adding animal terms).
- About 14 policy, regulation and governance papers (dark-sky governance, lighting standards, public perception): possibly the "units to legal issues" thread.
- A few light-measurement papers (nighttime Earth observation, city light pollution quantification, sensors).
- Duplicate records exist (same title twice): add a de-duplication step.
- Some retrieved papers never mention light (equestrian business, malaria, photography): a "mentions light" flag would catch them.

## 6. Save the table

Run this after the scope step so the saved CSV includes the `human_health` and `scope` columns.

In [10]:
df.to_csv("../data/output/table_abstracts.csv", index=False)


## Appendix A: seed KG overview (context only)

Not part of the mining flow. Shows how the 40 hand-entered triples in `kg/seed_statements.csv` map onto items, and which items are classified in `kg/items.csv`.

In [11]:
kg = pd.read_csv("../kg/seed_statements.csv")

items = pd.DataFrame({"item": sorted(set(kg["subject"]) | set(kg["object"]))})

def roles(name):
    as_subject = sorted(set(kg.loc[kg["subject"] == name, "property"]))
    as_object = sorted(set(kg.loc[kg["object"] == name, "property"]))
    return pd.Series({"subject_of": ", ".join(as_subject), "object_of": ", ".join(as_object)})

items = pd.concat([items, items["item"].apply(roles)], axis=1)
print(len(items))
items

27


,item,subject_of,object_of
0,V-band,,given_in_band
1,W/m2,,reported_in
2,W/m2/nm,,reported_in
3,W/m2/sr,,reported_in
4,W/m2/sr/nm,,reported_in
5,astronomers,,used_by
6,biologists,,used_by
7,cd/m2,,reported_in
8,ecologists,,used_by
9,illuminance,"photometric_version_of, reported_in, used_by",can_derive


In [12]:
items_cls = pd.read_csv("../kg/items.csv")
print(items_cls.shape)
print(items_cls["class"].value_counts())
known = set(kg["subject"]) | set(kg["object"])
print("not found in seed_statements:", sorted(set(items_cls["item"]) - known))



(11, 2)
class
quantity         8
spectral band    2
instrument       1
Name: count, dtype: int64
not found in seed_statements: []


In [13]:
target = df[(df["scope"] == "in scope") & (df[list(categories)] == "").all(axis=1)]
print(len(target), "in-scope papers with no match")

for word in ["species", "circadian"]:
    hits = target[target["text"].str.contains(word, case=False)]
    print(word, "-", len(hits), "papers")
    for t in hits["title"].sample(min(8, len(hits)), random_state=1):
        print("  -", t)
    print()

274 in-scope papers with no match
species - 40 papers
  - The influence of artificial light at night on a macroalgal species in Cabo Verde in a global comparison
  - Nocturnal Behavior of Wintering Common Cranes ( Grus grus ) Under Varying Environmental Conditions
  - Artificial light at night and invasive signal crayfish alter aquatic‐terrestrial food webs
  - Facultative brightness-change in an invasive gecko entering a novel background environment
  - Substrate heterogeneity outweighs colour in shaping thermal environment and intertidal barnacle recruitment on artificial surfaces
  - Reproductive stage affects the daily behavioral patterns of parthenogenetic marbled crayfish
  - Global change refugia could shelter species from multiple threats
  - Developing a Biodiversity Assessment Tool to Enable the Integration of Biodiversity into Sustainable Building Practice: The New Zealand Biodiversity Factor—Building (NZBF-B)

circadian - 16 papers
  - Impact of Artificial Night Lights on N

In [14]:
for t in target["title"].sample(30, random_state=7):
    print("-", t)

- Effect of monobutyrate on growth performance, intestinal development, and cecum microbiota in broilers
- Taxonomic and functional responses of benthic and drifting macroinvertebrates to fine sediment deposition: evidence from an alpine flume-based experiment
- Threadworms navigate with a carbon dioxide compass
- Conducting Impactful Research at Primarily Undergraduate Institutions
- Kajian Kebijakan Langit Gelap untuk Pembangunan Berkelanjutan
- Integrated Solar Energy Utilization and Conduction Filtration System—Medusozoa
- Is urban part-time lighting a mitigation measure for a common amphibian? A case study on Bufo spinosus
- Ocean Rafting: Marine Litter and Benthic Stopovers Amplify Species Dispersal Opportunities
- Resident survey on nocturnal environments and artificial light at night in the Brière region, France: data and replication materials
- Substrate heterogeneity outweighs colour in shaping thermal environment and intertidal barnacle recruitment on artificial surfaces
- L

In [15]:
# from collections import Counter

# stop = set("""the and of in to a is for on with that this are was were by as an be at
# from or it its their these those which we our has have had not but also can may more
# than been between among during under over into using used use results study studies
# effects effect based however both such other new high low two one all""".split())

# generic = {"light", "artificial", "night", "lighting", "pollution", "alan", "nighttime"}

# target = df[no_match & (df["scope"] == "in scope")]

# counts = Counter()
# for text in target["text"]:
#     words = set(re.findall(r"[a-z][a-z-]{3,}", text.lower()))
#     counts.update(words - stop - generic)

# pd.DataFrame(counts.most_common(100), columns=["word", "papers"])
# for word in ["species", "circadian"]:
#     hits = target[target["text"].str.contains(word, case=False)]
#     print(word, "-", len(hits), "papers")
#     for t in hits["title"].sample(8, random_state=1):
#         print("  -", t)
#     print()

## Appendix B: scratch snippets (commented out)

Uncomment to use. The first shows a random sample of papers that matched nothing; the second prints full abstracts for chosen row numbers.

In [16]:
# df[no_match][["title", "abstract"]].sample(10, random_state=1)

In [17]:
# for i in [144, 1716, 940]:
#     print(i, df.loc[i, "title"])
#     print(df.loc[i, "abstract"])
#     print()